In [0]:
%pip install langchain langchain-openai langchain-databricks
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# ---- imports.py ----
import os
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
from langchain_databricks import DatabricksVectorSearch

OPENAI_API_KEY = dbutils.secrets.get(scope="all", key="openaiapi")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

ENDPOINT_NAME = "export_ai_vector_endpoint"
INDEX_NAME    = "export_ai.rag.document_chunks_index"

print("✅ RAG chain imports done")

✅ RAG chain imports done


In [0]:
# ---- load_retriever.py ----
def load_retriever():
    vectorstore = DatabricksVectorSearch(
        endpoint   = ENDPOINT_NAME,
        index_name = INDEX_NAME,
        columns    = ["content", "source", "type"]
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 5}
    )
    print("✅ Retriever loaded")
    return retriever

retriever = load_retriever()

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True to VectorSearchClient().
✅ Retriever loaded


In [0]:
# ---- prompt.py ----
SYSTEM_PROMPT = """
You are an expert Indian export trade advisor helping Indian exporters.

You help with:
- Legal documents required to export to specific countries
- Country specific trade regulations and guidelines
- Why certain countries are recommended for specific products
- Follow up questions about previously recommended countries

When asked "why this country?" or similar questions, use the 
opportunity score data provided to explain in simple terms:
- Low tariffs mean cheaper entry into the market
- Close distance means lower shipping costs
- Strong currency means better payment value
- Large market size means more demand

Always answer in clear simple language suitable for small 
Indian exporters who may not know trade jargon.

If you don't know something, say so honestly.

Context from documents:
{context}

Chat History:
{chat_history}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template=SYSTEM_PROMPT
)

print("✅ Prompt template ready")

✅ Prompt template ready


In [0]:
# ---- rag_chain.py ----
def build_rag_chain(retriever):
    llm = ChatOpenAI(
        model      = "gpt-4o-mini",
        temperature= 0.3
    )

    memory = ConversationBufferMemory(
        memory_key    = "chat_history",
        return_messages= True,
        output_key    = "answer"
    )

    chain = ConversationalRetrievalChain.from_llm(
        llm                       = llm,
        retriever                 = retriever,
        memory                    = memory,
        combine_docs_chain_kwargs = {"prompt": prompt},
        return_source_documents   = True,
        verbose                   = False
    )

    print("✅ RAG chain built")
    return chain

chain = build_rag_chain(retriever)

✅ RAG chain built


/home/spark-da96fd0f-1728-4fc4-a89a-e1/.ipykernel/10109/command-7553042186068933-3971986367:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [0]:
# ---- test_rag_chain.py ----
def ask(chain, question):
    result  = chain({"question": question})
    answer  = result["answer"]
    sources = list(set([
        doc.metadata.get("source", "unknown")
        for doc in result["source_documents"]
    ]))
    print(f"\n🙋 Question : {question}")
    print(f"🤖 Answer   : {answer}")
    print(f"📄 Sources  : {sources}")
    return answer

# Test basic question
ask(chain, "What documents are needed to export food products to Bahrain?")

/home/spark-da96fd0f-1728-4fc4-a89a-e1/.ipykernel/10109/command-7553042186068934-3382223135:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result  = chain({"question": question})


---------------------------------------------------------------------------
RateLimitError                            Traceback (most recent call last)
File <command-7553042186068934>, line 15
     12     return answer
     14 # Test basic question
---> 15 ask(chain, "What documents are needed to export food products to Bahrain?")

File <command-7553042186068934>, line 3, in ask(chain, question)
      2 def ask(chain, question):
----> 3     result  = chain({"question": question})
      4     answer  = result["answer"]
      5     sources = list(set([
      6         doc.metadata.get("source", "unknown")
      7         for doc in result["source_documents"]
      8     ]))

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-da96fd0f-1728-4fc4-a89a-e14a2b0aa444/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:193, in deprecated.<locals>.deprecate.<locals>.warning_emitting_wrapper(*args, **kwargs)
    191     warned = True
    192     emit_warning()
--> 193 return wrapped(*arg

In [0]:
# ---- test_followup.py ----

# Test conversation memory with follow up questions
ask(chain, "What documents are needed to export spices to Bahrain?")
ask(chain, "Why is Bahrain a good market?")
ask(chain, "What about Greece?")
ask(chain, "Which of these two is better?")

---------------------------------------------------------------------------
RateLimitError                            Traceback (most recent call last)
File <command-7553042186068935>, line 4
      1 # ---- test_followup.py ----
      2 
      3 # Test conversation memory with follow up questions
----> 4 ask(chain, "What documents are needed to export spices to Bahrain?")
      5 ask(chain, "Why is Bahrain a good market?")
      6 ask(chain, "What about Greece?")

File <command-7553042186068934>, line 3, in ask(chain, question)
      2 def ask(chain, question):
----> 3     result  = chain({"question": question})
      4     answer  = result["answer"]
      5     sources = list(set([
      6         doc.metadata.get("source", "unknown")
      7         for doc in result["source_documents"]
      8     ]))

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-da96fd0f-1728-4fc4-a89a-e14a2b0aa444/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:193, in deprecated.<locals>.deprec